# IoT Edge-Cloud Data Reduction Framework
## Tutorial and Demonstration

This notebook provides a step-by-step guide to using the IoT edge-cloud data reduction framework.

**Author:** Aman Sharma  
**Date:** January 2025

## 1. Setup and Imports

First, let's import the necessary modules and set up our environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import sys
import os

# Add parent directory to path if needed
# sys.path.append('..')

# Configure matplotlib for better plots
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Setup complete")

## 2. Generate Synthetic Sensor Data

Let's start by generating synthetic IoT sensor data to work with.

In [ ]:
from data_generation.synthetic_sensors import generate_dataset

# Generate data for 3 sensors over 500 timesteps
n_sensors = 3
timesteps = 500
noise_level = 0.08

data = generate_dataset(
    n_sensors=n_sensors,
    T=timesteps,
    noise=noise_level,
    spikes=True,
    drift=0.05,
    seed=42
)

print(f"Generated data shape: {data.shape}")
print(f"Data type: {data.dtype}")

# Visualize the raw sensor data
plt.figure(figsize=(14, 5))
for i in range(n_sensors):
    plt.plot(data[i], label=f'Sensor {i}', alpha=0.8)
plt.xlabel('Time Step')
plt.ylabel('Temperature (°C)')
plt.title('Raw Sensor Data')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Apply Data Reduction Strategies

Now let's apply different edge reduction strategies to see how they work.

### 3.1 Adaptive Sampling

In [ ]:
from edge_reduction.adaptive_sampling import AdaptiveSampler
from evaluation.accuracy_metrics import reconstruct_from_samples, rmse

# Create sampler
sampler = AdaptiveSampler(
    base_interval=1.0,
    low_thresh=0.12,
    high_thresh=0.35
)

# Apply to first sensor
sensor_0 = data[0]
# Handle NaN values
sensor_0_clean = np.nan_to_num(sensor_0, nan=0.0)

sampled_idx, sampled_vals = sampler.run(sensor_0_clean)

print(f"Original samples: {len(sensor_0_clean)}")
print(f"Kept samples: {len(sampled_idx)}")
print(f"Reduction: {100*(1 - len(sampled_idx)/len(sensor_0_clean)):.1f}%")

# Reconstruct and measure error
reconstructed = reconstruct_from_samples(len(sensor_0_clean), sampled_idx, sampled_vals)
error = rmse(sensor_0_clean, reconstructed)
print(f"Reconstruction RMSE: {error:.4f}")

# Visualize
plt.figure(figsize=(14, 5))
plt.plot(sensor_0_clean, 'b-', label='Original', alpha=0.5)
plt.plot(sampled_idx, sampled_vals, 'ro', markersize=4, label='Sampled Points')
plt.plot(reconstructed, 'g--', label='Reconstructed', alpha=0.8)
plt.xlabel('Time Step')
plt.ylabel('Temperature')
plt.title('Adaptive Sampling Results')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.2 Event-Driven (Adaptive Threshold)

In [ ]:
from edge_reduction.event_driven import AdaptiveThresholdReducer

# Create reducer
reducer = AdaptiveThresholdReducer(
    initial_thresh=0.22,
    max_gap=40
)

# Apply to first sensor
event_idx, event_vals = reducer.run(sensor_0_clean)

print(f"Original samples: {len(sensor_0_clean)}")
print(f"Transmitted samples: {len(event_idx)}")
print(f"Reduction: {100*(1 - len(event_idx)/len(sensor_0_clean)):.1f}%")

# Reconstruct and measure error
reconstructed_event = reconstruct_from_samples(len(sensor_0_clean), event_idx, event_vals)
error_event = rmse(sensor_0_clean, reconstructed_event)
print(f"Reconstruction RMSE: {error_event:.4f}")

# Visualize
plt.figure(figsize=(14, 5))
plt.plot(sensor_0_clean, 'b-', label='Original', alpha=0.5)
plt.plot(event_idx, event_vals, 'mo', markersize=4, label='Transmitted Points')
plt.plot(reconstructed_event, 'g--', label='Reconstructed', alpha=0.8)
plt.xlabel('Time Step')
plt.ylabel('Temperature')
plt.title('Event-Driven (Adaptive Threshold) Results')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.3 Aggregation

In [ ]:
from edge_reduction.aggregation import aggregate_array

# Apply aggregation
window = 5
aggregated = aggregate_array(sensor_0_clean, window=window, method='mean')
agg_idx = np.arange(window-1, len(sensor_0_clean), window)
if len(agg_idx) > len(aggregated):
    agg_idx = agg_idx[:len(aggregated)]

print(f"Original samples: {len(sensor_0_clean)}")
print(f"Aggregated samples: {len(aggregated)}")
print(f"Reduction: {100*(1 - len(aggregated)/len(sensor_0_clean)):.1f}%")

# Visualize
plt.figure(figsize=(14, 5))
plt.plot(sensor_0_clean, 'b-', label='Original', alpha=0.5)
plt.plot(agg_idx, aggregated, 'ro-', markersize=6, label=f'Aggregated (window={window})')
plt.xlabel('Time Step')
plt.ylabel('Temperature')
plt.title('Aggregation Results')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Compare All Strategies

Let's compare all reduction strategies side by side.

In [ ]:
# Compile results
results = {
    'Strategy': ['Adaptive Sampling', 'Event-Driven', 'Aggregation'],
    'Samples Kept': [len(sampled_idx), len(event_idx), len(aggregated)],
    'Reduction %': [
        100*(1 - len(sampled_idx)/len(sensor_0_clean)),
        100*(1 - len(event_idx)/len(sensor_0_clean)),
        100*(1 - len(aggregated)/len(sensor_0_clean))
    ],
    'RMSE': [error, error_event, np.nan]  # Aggregation doesn't have direct reconstruction
}

results_df = pd.DataFrame(results)
print("\n=== Comparison Table ===")
print(results_df.to_string(index=False))

# Bar chart comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Reduction percentage
ax1.bar(results_df['Strategy'], results_df['Reduction %'], color=['skyblue', 'lightcoral', 'lightgreen'])
ax1.set_ylabel('Data Reduction (%)')
ax1.set_title('Reduction Efficiency')
ax1.grid(axis='y', alpha=0.3)

# RMSE
ax2.bar(results_df['Strategy'][:2], results_df['RMSE'][:2], color=['skyblue', 'lightcoral'])
ax2.set_ylabel('RMSE')
ax2.set_title('Reconstruction Error')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Running Full Experiments

The framework includes automated parameter sweeps and statistical analysis.

In [ ]:
# Load configuration
with open('data_generation/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Current configuration:")
print(f"  Mode: {config['experiment']['mode']}")
print(f"  Runs: {config['experiment']['runs']}")
print(f"  Sensors: {config['experiment']['sensors']}")
print(f"  Timesteps: {config['experiment']['timesteps']}")
print(f"  Reducers: {config['experiment']['reducers']}")
print(f"  Serializers: {config['experiment']['serializers']}")

To run a full experiment, execute:

```python
!python main.py
```

This will generate:
- `results/results_vectorized.csv` - Raw results
- `results/summary_table.csv` - Summary statistics
- `results/fig_tradeoff.png` - Trade-off plot

## 6. Analyzing Results

Once you've run experiments, you can analyze the results.

In [ ]:
# Load results (if available)
try:
    results_df = pd.read_csv('results/results_vectorized.csv')
    print(f"Loaded {len(results_df)} result rows")
    print("\nFirst few rows:")
    print(results_df.head())
    
    # Summary statistics
    print("\n=== Summary by Reducer (JSON) ===")
    summary = results_df[results_df['serializer']=='JSON'].groupby('reducer').agg({
        'reduction_pct': 'mean',
        'rmse_mean': 'mean',
        'bytes': 'mean',
        'energy_proxy': 'mean'
    }).round(2)
    print(summary)
    
except FileNotFoundError:
    print("No results found. Run 'python main.py' first to generate results.")

## 7. Advanced Visualization

Use the enhanced visualization module for publication-quality plots.

In [ ]:
from evaluation.visualizer import ExperimentVisualizer

try:
    results_df = pd.read_csv('results/results_vectorized.csv')
    visualizer = ExperimentVisualizer(results_df, output_dir='results')
    
    # Generate all standard plots
    visualizer.generate_all_plots(show=True)
    
    print("✓ All plots generated!")
except FileNotFoundError:
    print("No results found. Run experiments first.")

## 8. Database Analysis

If you've run streaming experiments with database storage, you can analyze the stored data.

In [ ]:
from cloud.analyzer import DataAnalyzer

try:
    with DataAnalyzer('results/cloud_data.db') as analyzer:
        print("=== Database Overview ===")
        print(f"Total records: {analyzer.get_total_records()}")
        print(f"Unique sensors: {analyzer.get_sensor_count()}")
        
        print("\n=== Per-Sensor Statistics ===")
        sensor_stats = analyzer.get_sensor_stats()
        print(sensor_stats.to_string())
        
        print("\n=== Data Quality Report ===")
        quality = analyzer.get_data_quality_report()
        for key, value in quality.items():
            print(f"  {key}: {value}")
except FileNotFoundError:
    print("No database found. Run streaming mode with receive_to_db=true first.")

## 9. Configuration Validation

Always validate your configuration before running experiments.

In [ ]:
from data_generation.config_validator import ConfigValidator

validator = ConfigValidator('data_generation/config.yaml')
is_valid, errors, warnings = validator.validate(raise_on_error=False)

validator.print_report()

if is_valid:
    print("\n✓ Configuration is valid and ready for experiments!")
else:
    print("\n✗ Please fix configuration errors before running experiments.")

## 10. Next Steps

Now that you understand the basics, you can:

1. **Modify parameters** in `config.yaml` to test different settings
2. **Add new reduction strategies** by creating modules in `edge_reduction/`
3. **Run parameter sweeps** to find optimal configurations
4. **Compare serialization formats** (JSON vs CBOR)
5. **Test with real IoT data** by adapting the data generation module

### Useful Commands:

```bash
# Run full experiment
python main.py

# Validate configuration
python data_generation/config_validator.py

# Generate visualizations
python evaluation/visualizer.py

# Analyze database
python cloud/analyzer.py
```

### Documentation:

See `README.md` for complete documentation and examples.